# 🏆 AIML Hackathon 2526 — Pareto Team Solution

**1st place** on the public leaderboard with MAP@10 = **0.7268667247**.

## Authors (equal contribution)

- **Lorenzo Cerovaz** — [@CerovazS](https://github.com/CerovazS)
- **Federico Forner** — [@Fede2717](https://github.com/Fede2717)
- **Marco Galletti** — [@m4rch1n0](https://github.com/m4rch1n0)

Two-stage neural passage ranking on a sampled subset of MS MARCO:
1. **First stage** — Dense retrieval with `all-mpnet-base-v2` + FAISS (top-300 candidates per query).
2. **Second stage** — Cross-Encoder ensemble (BGE-M3, Jina v2, MiniLM-L12) fused via unweighted Reciprocal Rank Fusion with smoothing $k=1$.

See the [repository README](https://github.com/m4rch1n0/aiml-hackathon-2526-passage-ranking) and [`paper.pdf`](./paper.pdf) for the full method, ablations, and the simpler-generalizes-better finding.

---

In [ ]:
!pip install torch transformers sentence-transformers faiss-cpu tqdm pandas numpy pyarrow -q

In [ ]:
import os
import zipfile
import urllib.request

DATASET_URL = "https://github.com/fabsilvestri/aiml_hackathon_data/releases/download/v1.0/kaggle_data.zip"
DATA_DIR = "data"

print("Downloading dataset...")
urllib.request.urlretrieve(DATASET_URL, "kaggle_data.zip")
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile("kaggle_data.zip", "r") as zf:
    zf.extractall(DATA_DIR)
os.remove("kaggle_data.zip")
print("Dataset ready!")


In [ ]:
import gc
from collections import defaultdict
from typing import Dict, List, Optional

import faiss
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification

# CONFIGURATION

CONFIG = {
    # Device
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    
    # Data paths
    "data_dir": "data",
    "collection_path": "data/collection.parquet",
    "test_path": "data/test.csv",
    
    # First-stage retrieval (Dual Encoder)
    "retrieval": {
        "model_name": "sentence-transformers/all-mpnet-base-v2",
        "batch_size": 64,
        "max_length": 256,
        "normalize": True,
        "query_prefix": "",
        "passage_prefix": "",
        "top_k": 300,  # Candidates for reranking
    },
    
    # Second-stage reranking (Cross-Encoders Ensemble)
    "reranking": {
        "batch_size": 4,
        "max_length": 512,
        "rrf_k": 1,  # RRF smoothing parameter
        "models": [
            "BAAI/bge-reranker-v2-m3",
            "jinaai/jina-reranker-v2-base-multilingual",
            "cross-encoder/ms-marco-MiniLM-L-12-v2",
        ],
    },
    
    # Output
    "output_path": "submission.csv",
    "top_k_submission": 10,  # MAP@10 evaluation
}

print(f"Device: {CONFIG['device']}")
print(f"Retrieval model: {CONFIG['retrieval']['model_name']}")
print(f"Reranking models: {len(CONFIG['reranking']['models'])}")
print(f"Top-K candidates: {CONFIG['retrieval']['top_k']}")


In [ ]:
class DualEncoderRetriever:
    """
    Dual Encoder (Bi-Encoder) retriever with FAISS for efficient similarity search.
    
    Architecture:
        Query -> Encoder -> q (vector) -> dot(q, p) -> score
        Passage -> Encoder -> p (vector)
    """
    
    def __init__(
        self,
        model_name: str = "sentence-transformers/all-mpnet-base-v2",
        query_prefix: str = "",
        passage_prefix: str = "",
        normalize: bool = True,
        max_length: int = 256,
        batch_size: int = 64,
        device: str = "cuda",
    ):
        self.model_name = model_name
        self.query_prefix = query_prefix
        self.passage_prefix = passage_prefix
        self.normalize = normalize
        self.max_length = max_length
        self.batch_size = batch_size
        self.device = device
        
        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        
        # Embedding dimension
        self.embed_dim = self.model.config.hidden_size
        print(f"Embedding dim: {self.embed_dim}")
        
        # Will be populated after indexing
        self.pids: Optional[np.ndarray] = None
        self.index = None
        self.is_indexed = False
    
    @torch.no_grad()
    def _encode(
        self,
        texts: List[str],
        prefix: str = "",
        show_progress: bool = True,
        desc: str = "Encoding",
    ) -> np.ndarray:
        """
        Encode texts into embeddings using mean pooling.
        """
        # Add prefix
        if prefix:
            texts = [prefix + t for t in texts]
        
        all_embeddings = []
        
        iterator = range(0, len(texts), self.batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc=desc, unit="batch")
        
        for i in iterator:
            batch_texts = texts[i : i + self.batch_size]
            
            # Tokenize
            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}
            
            # Forward pass
            outputs = self.model(**encoded)
            
            # Mean pooling
            attention_mask = encoded["attention_mask"]
            token_embeddings = outputs.last_hidden_state
            
            mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            embeddings = sum_embeddings / sum_mask
            
            # Normalize if requested
            if self.normalize:
                embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            
            all_embeddings.append(embeddings.cpu().numpy())
        
        return np.vstack(all_embeddings).astype(np.float32)
    
    def index_passages(
        self,
        collection: pd.DataFrame,
        pid_col: str = "pid",
        passage_col: str = "passage",
    ) -> None:
        """
        Index all passages in the collection using FAISS.
        """
        print(f"\n{'='*60}")
        print("INDEXING PASSAGES")
        print(f"{'='*60}")
        print(f"Collection size: {len(collection):,}")
        
        # Store PIDs
        self.pids = collection[pid_col].astype(str).values
        
        # Encode passages
        print(f"Encoding {len(collection):,} passages...")
        passages = collection[passage_col].tolist()
        embeddings = self._encode(passages, prefix=self.passage_prefix, desc="Indexing passages")
        
        # Build FAISS index (Flat Inner Product)
        print(f"Building FAISS index (flat_ip)...")
        self.index = faiss.IndexFlatIP(self.embed_dim)
        self.index.add(embeddings)
        self.is_indexed = True
        
        print(f"Index size: {self.index.ntotal:,} vectors")
        print(f"{'='*60}\n")
    
    @torch.no_grad()
    def retrieve(
        self,
        queries: List[str],
        query_ids: List[str],
        top_k: int = 300,
    ) -> Dict[str, List[str]]:
        """
        Retrieve top-k passages for each query.
        
        Returns:
            Dict[qid -> List[pid]] ordered by relevance score (descending)
        """
        if not self.is_indexed:
            raise RuntimeError("Index not built. Call index_passages() first.")
        
        print(f"\n{'='*60}")
        print("RETRIEVAL")
        print(f"{'='*60}")
        print(f"Queries: {len(queries):,}")
        print(f"Top-K: {top_k:,}")
        
        # Encode queries
        print("Encoding queries...")
        query_embeddings = self._encode(queries, prefix=self.query_prefix, desc="Encoding queries")
        
        # Search with FAISS
        print("Searching FAISS index...")
        scores, indices = self.index.search(query_embeddings, top_k)
        
        # Build results
        results = {}
        for i, qid in enumerate(query_ids):
            results[qid] = [
                self.pids[idx]
                for idx in indices[i]
                if idx >= 0  # FAISS returns -1 if not enough results
            ]
        
        print(f"Retrieved candidates for {len(results)} queries")
        print(f"{'='*60}\n")
        return results
    
    def cleanup(self):
        """Free GPU memory."""
        del self.model
        del self.tokenizer
        torch.cuda.empty_cache()
        gc.collect()
        print("Retriever cleaned up")

print("DualEncoderRetriever class defined")


In [ ]:
class CrossEncoderReranker:
    """
    Cross-Encoder reranker for second-stage ranking.
    
    Architecture:
        [CLS] Query [SEP] Passage [SEP] -> Encoder -> Score
    """
    
    def __init__(
        self,
        model_name: str,
        batch_size: int = 4,
        max_length: int = 512,
        device: str = "cuda",
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length
        self.device = device
        
        # Load model
        print(f"Loading reranker: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, trust_remote_code=True, torch_dtype="auto"
        ).to(device)
        self.model.eval()
        print(f"Model loaded")
    
    @torch.no_grad()
    def score_pairs(
        self,
        query: str,
        passages: List[str],
    ) -> List[float]:
        """
        Score query-passage pairs.
        
        Returns:
            List of scores for each passage
        """
        if not passages:
            return []
        
        pairs = [[query, passage] for passage in passages]
        all_scores = []
        
        for i in range(0, len(pairs), self.batch_size):
            batch = pairs[i : i + self.batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            ).to(self.device)
            
            outputs = self.model(**inputs)
            scores = outputs.logits.view(-1).float().cpu().numpy()
            all_scores.extend(scores.tolist())
        
        return all_scores
    
    def cleanup(self):
        """Free GPU memory."""
        del self.model
        del self.tokenizer
        torch.cuda.empty_cache()
        gc.collect()


print("CrossEncoderReranker class defined")


In [ ]:
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

# Load collection
collection = pd.read_parquet(CONFIG["collection_path"])
print(f"Collection: {len(collection):,} passages")

# Create PID -> Passage mapping
pid2passage = dict(zip(collection["pid"].astype(str), collection["passage"]))

# Load test queries
test_df = pd.read_csv(CONFIG["test_path"])
print(f"Test queries: {len(test_df):,}")

# Create QID -> Query mapping
qid2query = dict(zip(test_df["id"].astype(str), test_df["query"]))
query_ids = list(qid2query.keys())
queries = list(qid2query.values())

print(f"\nData loaded successfully")
print(f"Expected submission rows: {len(test_df)}")


In [ ]:
print("\n" + "="*60)
print("PHASE 1: FIRST-STAGE RETRIEVAL")
print("="*60)

# Initialize retriever
retriever = DualEncoderRetriever(
    model_name=CONFIG["retrieval"]["model_name"],
    query_prefix=CONFIG["retrieval"]["query_prefix"],
    passage_prefix=CONFIG["retrieval"]["passage_prefix"],
    normalize=CONFIG["retrieval"]["normalize"],
    max_length=CONFIG["retrieval"]["max_length"],
    batch_size=CONFIG["retrieval"]["batch_size"],
    device=CONFIG["device"],
)

# Index passages
retriever.index_passages(collection)

# Retrieve candidates
retrieved_candidates = retriever.retrieve(
    queries=queries,
    query_ids=query_ids,
    top_k=CONFIG["retrieval"]["top_k"],
)

# Cleanup to free memory for rerankers
retriever.cleanup()
del retriever

print(f"Retrieved {CONFIG['retrieval']['top_k']} candidates per query")


In [ ]:
print("\n" + "="*60)
print("PHASE 2: CROSS-ENCODER ENSEMBLE RERANKING")
print("="*60)

RERANKING_CONFIG = CONFIG["reranking"]
RRF_K = RERANKING_CONFIG["rrf_k"]
ELITE_MODELS = RERANKING_CONFIG["models"]

print(f"Models: {len(ELITE_MODELS)}")
print(f"RRF K: {RRF_K}")
print(f"Batch size: {RERANKING_CONFIG['batch_size']}")

# Accumulate RRF scores across all models
# final_rrf_scores[qid][pid] = accumulated RRF score
final_rrf_scores = defaultdict(lambda: defaultdict(float))

for model_idx, model_name in enumerate(ELITE_MODELS):
    print(f"\n{'-'*50}")
    print(f"Model {model_idx + 1}/{len(ELITE_MODELS)}: {model_name}")
    print(f"{'-'*50}")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    gc.collect()
    
    # Load reranker
    try:
        reranker = CrossEncoderReranker(
            model_name=model_name,
            batch_size=RERANKING_CONFIG["batch_size"],
            max_length=RERANKING_CONFIG["max_length"],
            device=CONFIG["device"],
        )
    except Exception as e:
        print(f"Error loading {model_name}: {e}")
        continue
    
    # Score all queries
    for qid in tqdm(query_ids, desc=f"Scoring {model_name.split('/')[-1]}"):
        if qid not in retrieved_candidates:
            continue
        
        candidates = retrieved_candidates[qid]
        query_text = qid2query[qid]
        
        # Get passages for valid PIDs
        valid_pids = [pid for pid in candidates if pid in pid2passage]
        passages = [pid2passage[pid] for pid in valid_pids]
        
        if not passages:
            continue
        
        # Score pairs
        scores = reranker.score_pairs(query_text, passages)
        
        # Sort by score (descending) to get ranking
        scored_pids = list(zip(valid_pids, scores))
        scored_pids.sort(key=lambda x: x[1], reverse=True)
        
        # Apply RRF scoring (accumulate across models)
        for rank, (pid, _) in enumerate(scored_pids):
            rrf_score = 1.0 / (RRF_K + rank + 1)
            final_rrf_scores[qid][pid] += rrf_score
    
    # Cleanup model
    reranker.cleanup()
    del reranker

print(f"\nEnsemble reranking completed")


In [ ]:
print("\n" + "="*60)
print("PHASE 3: GENERATE SUBMISSION")
print("="*60)

TOP_K = CONFIG["top_k_submission"]
submission_rows = []

for qid in query_ids:
    if qid in final_rrf_scores:
        # Get accumulated RRF scores from ensemble
        candidates_scores = final_rrf_scores[qid]
        # Sort by total RRF score (descending)
        sorted_candidates = sorted(candidates_scores.items(), key=lambda x: x[1], reverse=True)
        # Take top-10 PIDs
        top_pids = [pid for pid, _ in sorted_candidates[:TOP_K]]
    else:
        # Fallback: use retrieval results directly
        if qid in retrieved_candidates:
            top_pids = retrieved_candidates[qid][:TOP_K]
        else:
            top_pids = []
    
    # Create space-separated string
    prediction_string = " ".join(top_pids)
    submission_rows.append({"id": qid, "expected": prediction_string})

# Create DataFrame
submission_df = pd.DataFrame(submission_rows)

# Validation
print(f"\nSubmission Statistics:")
print(f"Total rows: {len(submission_df)}")
print(f"Expected rows: 4912")

if len(submission_df) != 4912:
    print(f"WARNING: Row count mismatch! Expected 4912, got {len(submission_df)}")
else:
    print(f"Row count matches")

# Save submission
submission_df.to_csv(CONFIG["output_path"], index=False)
print(f"\nSubmission saved to: {CONFIG['output_path']}")

# Show sample
print(f"\nSample submission (first 3 rows):")
print(submission_df.head(3).to_string(index=False))


In [ ]:
print("\n" + "="*60)
print("FINAL VERIFICATION")
print("="*60)

# Re-read and verify submission
final_check = pd.read_csv(CONFIG["output_path"])

print(f"\nSubmission file check:")
print(f"Columns: {list(final_check.columns)}")
print(f"Rows: {len(final_check)}")
print(f"Header: id,expected (correct)" if list(final_check.columns) == ["id", "expected"] else "Wrong header!")

# Check for empty predictions
empty_count = (final_check["expected"] == "").sum()
print(f"Empty predictions: {empty_count}")

# Check average number of predictions per row
avg_preds = final_check["expected"].apply(lambda x: len(str(x).split()) if pd.notna(x) and x != "" else 0).mean()
print(f"Avg predictions per query: {avg_preds:.1f}")

print(f"\n{'='*60}")
print("SUBMISSION READY")
print("Download 'submission.csv' and upload to Kaggle")
print(f"{'='*60}")
